# M02C · Biopython: SeqIO, GenBank & SeqUtils
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2024-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M02_File_Handling/solutions/M02C_Biopython_SeqIO_GenBank_Solutions.ipynb)

⏱ ~120 min · Difficulty: ★★☆ · Prerequisites: M02A (Text Files), M02B (FASTA & FASTQ by Hand)

---


## 1. Learning Outcomes

By the end of this notebook, you will be able to:
1. **Navigate** the Biopython package hierarchy and manipulate `Bio.Seq` objects (transcription, translation with genetic code tables, reverse complementation).
2. **Construct and inspect** `Bio.SeqRecord` objects containing biological sequences, accession metadata, feature annotations, and per-letter quality metrics.
3. **Compare** `SeqIO.read()`, `SeqIO.parse()`, `SeqIO.to_dict()`, and `SeqIO.index()` for reading single vs multi-record files.
4. **Query GenBank records** to inspect organism taxonomy and traverse genomic features (`CDS`, `gene`), locations, strand orientation, and qualifiers.
5. **Compute** biochemical properties including molecular weight, GC fraction, melting temperature, and protein physicochemical parameters via `Bio.SeqUtils` and `Bio.SeqUtils.ProtParam`.
6. **Perform inter-conversion** between standard genomic formats (GenBank, FASTA, FASTQ) using `SeqIO.convert()`.

---


## 2. Prerequisites

Before starting this notebook, ensure you understand:
- Reading and writing files using `pathlib.Path` and context managers (M02A).
- Structure of FASTA, FASTQ, and header formats (M02B).
- Basic Python dictionaries, list comprehensions, and functions.
- Molecular biology concepts: transcription, translation, codon tables, CDS features, and GenBank files.

---


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path

# Detect whether we are running in Google Colab or locally
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install biopython
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root
    ROOT = Path.cwd().resolve()
    while not (ROOT / "data").exists() and ROOT.parent != ROOT:
        ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M02C"
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import Bio.SeqUtils as SeqUtils
from Bio.SeqUtils.ProtParam import ProteinAnalysis

print(f"Python interpreter : {sys.version.split()[0]}")
print(f"Biopython version  : {Bio.__version__}")
print(f"Repository ROOT    : {ROOT.resolve()}")
print(f"DATA_DIR exists    : {DATA_DIR.exists()} ({DATA_DIR})")
print(f"OUT_DIR ready      : {OUT_DIR.exists()} ({OUT_DIR})")


## 4. Concept Sections

### 4.1 Concept 1: The Biopython Ecosystem & Architecture

**Biopython** is the premier open-source Python library for computational molecular biology. Established in 1999, it provides battle-tested data structures, parsers for dozens of biological file formats, interfaces to NCBI BLAST and Entrez, and tools for structural bioinformatics.

#### Biopython Module Map
```
Bio
├── Seq          → Core biological sequence representation & operations
├── SeqRecord    → Sequences annotated with IDs, descriptions, and features
├── SeqIO        → High-level input/output for FASTA, FASTQ, GenBank, EMBL, etc.
├── SeqUtils     → GC content, molecular weight, melting temperatures, ProtParam
├── Entrez       → Direct programmatic queries to NCBI (PubMed, GenBank, Gene)
├── PDB          → Structural biology: parse PDB/mmCIF, calculate atomic distances
├── Align        → Pairwise and multiple sequence alignments (PairwiseAligner)
└── KEGG         → Metabolic pathway querying & REST API
```


In [ ]:
# Worked Example 4.1: Checking Biopython components
print(f"Biopython installation path: {Bio.__file__}")
print(f"SeqIO supported formats count: {len(SeqIO._FormatToWriter)} formats")
print("Sample supported formats     : 'fasta', 'fastq', 'genbank', 'embl', 'swiss', 'tab'")


#### ✏️ Try it 4.1: Inspect Biopython Version
Verify that `Bio.__version__` is at least `"1.80"`.


In [ ]:
# Try it 4.1: Biopython version check
version_parts = [int(p) for p in Bio.__version__.split(".")[:2]]
assert version_parts[0] >= 1 and version_parts[1] >= 80, f"Biopython version must be >= 1.80, got {Bio.__version__}"
print(f"Biopython {Bio.__version__} is ready ✔")


#### ⚠️ Common Mistakes with Biopython
- **Importing `biopython` instead of `Bio`**: The pip package name is `biopython`, but the Python import statement is always `import Bio` (capital 'B').
- **Using deprecated methods**: In Biopython 1.78+, `Bio.Alphabet` was removed and sequences became immutable. In Biopython 1.80+, `Bio.SeqUtils.GC` was deprecated in favor of `Bio.SeqUtils.gc_fraction`. Always use modern APIs.


### 4.2 Concept 2: The `Seq` Object

The `Bio.Seq.Seq` object is Biopython's fundamental sequence representation. It behaves like a Python string (supporting slicing, indexing, concatenation, and `len()`), but provides specialized biological methods:
- `.complement()`: Watson-Crick complement.
- `.reverse_complement()`: Standard 5′ $\rightarrow$ 3′ reverse complement.
- `.transcribe()`: DNA $\rightarrow$ RNA (replaces T with U).
- `.back_transcribe()`: RNA $\rightarrow$ DNA (replaces U with T).
- `.translate(table=1, to_stop=False)`: Translates codons to amino acids using NCBI genetic code tables.

#### Immutability
Since Biopython 1.78+, `Seq` objects are **immutable**, exactly like Python `str`. You cannot perform in-place item assignment (`seq[0] = 'G'`). To modify a sequence, convert it to a string or use `Bio.Seq.MutableSeq`.


In [ ]:
# Worked Example 4.2: Manipulating sequences with Seq
# Coding sequence of human TP53 partial exon
coding_dna = Seq("ATGGAGGAGCCGCAGTCAGATCCTAGCGTCGAGCCCCCTCTGAGTCAGGAAACATTTTCAGACCTATGGAAACTACTTCCTGAAAACAACGTTCTGTCCCCCTTGCCGTCCCAAGCAATGGATGATTTGATGCTGTCCCCGGACGATATTGAACAATGGTTCACTGAAGACCCAGGTCCAGATGAAGCTCCCAGAATGCCAGAGGCTGCTCCCCGCGTGGCCCCTGCACCAGCAGCTCCTACACCGGCGGCCCCTGCACCAGCCCCCTCCTGGCCC")

print(f"Sequence length        : {len(coding_dna)} bp")
print(f"Reverse complement     : {coding_dna.reverse_complement()[:30]}...")
print(f"mRNA transcription     : {coding_dna.transcribe()[:30]}...")

# Translation using Standard Genetic Code (Table 1)
protein = coding_dna.translate(table=1, to_stop=True)
print(f"Translated protein     : {protein}")
print(f"Protein length         : {len(protein)} amino acids")


#### ✏️ Try it 4.2: Bacterial Translation (Table 11)
Translate the bacterial sequence `"TTGAAACGTTAA"` using the bacterial genetic code (`table=11`, where `TTG` can serve as an alternate start codon) with `to_stop=True`.


In [ ]:
# Try it 4.2: Bacterial translation
bacterial_seq = Seq("TTGAAACGTTAA")
bacterial_protein = bacterial_seq.translate(table=11, to_stop=True)

print("Bacterial protein:", bacterial_protein)
assert str(bacterial_protein) == "LKR", f"Expected 'LKR', got {str(bacterial_protein)}"
print("Try it 4.2 passed ✔")


#### ⚠️ Common Mistakes with `Seq`
- **Attempting in-place modification**: Writing `seq[0] = "A"` raises `TypeError: 'Seq' object does not support item assignment`.
- **Forgetting `to_stop=True`**: By default, `.translate()` continues translating past stop codons, inserting `*` into the amino acid sequence. Use `to_stop=True` to truncate at the first stop codon.
- **Translating non-triplet lengths**: If `len(seq) % 3 != 0`, Biopython emits a `BiopythonWarning: Partial codon`.


### 4.3 Concept 3: The `SeqRecord` Object

While `Seq` stores raw sequence letters, `SeqRecord` encapsulates an entire biological record:
- `.seq`: The underlying `Seq` object.
- `.id`: The primary identifier (e.g., accession number `NM_000546.6`).
- `.name`: Short name or locus name (e.g., `TP53`).
- `.description`: Full human-readable definition line.
- `.annotations`: A Python dictionary of record-level metadata (e.g. `organism`, `taxonomy`, `date`, `molecule_type`).
- `.letter_annotations`: Per-letter information aligned with the sequence (e.g., Phred quality scores `phred_quality`).
- `.features`: A list of `SeqFeature` objects denoting genes, CDS, exons, and regulatory elements.


In [ ]:
# Worked Example 4.3: Constructing a SeqRecord from scratch
# Human Insulin B-chain partial sequence
insulin_b = Seq("FVNQHLCGSHLVEALYLVCGERGFFYTPKT")

rec = SeqRecord(
    seq=insulin_b,
    id="INS_B_HUMAN",
    name="Insulin_B",
    description="Human insulin chain B peptide",
    annotations={"organism": "Homo sapiens", "molecule_type": "protein"}
)

print(f"Record ID          : {rec.id}")
print(f"Description        : {rec.description}")
print(f"Molecule type      : {rec.annotations['molecule_type']}")
print(f"FASTA formatted representation:\n{rec.format('fasta')}")


#### ✏️ Try it 4.3: Custom SeqRecord Creation
Create a `SeqRecord` for *E. coli* beta-galactosidase partial sequence `Seq("TMITDSLAV")` with ID `"LACZ_ECOLI"`, and verify that `rec.format("fasta")` starts with `">LACZ_ECOLI"`.


In [ ]:
# Try it 4.3: Build SeqRecord
lacz_rec = SeqRecord(Seq("TMITDSLAV"), id="LACZ_ECOLI", description="lacZ partial")
fasta_repr = lacz_rec.format("fasta")

assert fasta_repr.startswith(">LACZ_ECOLI")
assert "TMITDSLAV" in fasta_repr
print("Try it 4.3 passed ✔")


#### ⚠️ Common Mistakes with `SeqRecord`
- **Modifying `.seq` length while `letter_annotations` exist**: If a record has quality scores in `.letter_annotations["phred_quality"]`, changing `.seq` without updating the scores raises `ValueError: Letter annotations must match sequence length`.
- **Confusing `rec.id` and `rec.description`**: When exporting to FASTA, Biopython places `rec.description` after `rec.id`. If `rec.description` already contains `rec.id`, you can end up with duplicated IDs in the header.


### 4.4 Concept 4: Reading Files with `SeqIO`: `read` vs `parse` vs `to_dict` vs `index`

`Bio.SeqIO` is the universal input/output engine. Choosing the right reading method is vital:

| Method | Syntax | Input Expectation | Return Type | Memory Complexity | Best Used For |
|---|---|---|---|---|---|
| **`SeqIO.read`** | `SeqIO.read(f, "genbank")` | **Exactly ONE** record | `SeqRecord` | Loads 1 record in RAM | Single-accession GenBank/FASTA files |
| **`SeqIO.parse`** | `SeqIO.parse(f, "fasta")` | **Zero or more** records | Generator yielding `SeqRecord` | **$O(1)$** streaming | Multi-FASTA, FASTQ, whole-genome assemblies |
| **`SeqIO.to_dict`** | `SeqIO.to_dict(SeqIO.parse(...))` | Multi-record | `dict[str, SeqRecord]` | Loads entire file into RAM | Fast in-memory key lookups (<500 MB) |
| **`SeqIO.index`** | `SeqIO.index(path, "fasta")` | Multi-record | Read-only dict-like index | **$O(1)$** (stores only byte offsets) | Random access in massive files (multi-GB) |

#### The `SeqIO.read()` Contract
If a file has 0 records, `SeqIO.read()` raises `ValueError: No records found`. If a file has 2 or more records, `SeqIO.read()` raises `ValueError: More than one record found`. Only use `.read()` when you are certain the file contains exactly one record!


In [ ]:
# Worked Example 4.4: SeqIO.read vs SeqIO.parse
# 1. SeqIO.read on single-record GenBank file
tp53_gb_path = DATA_DIR / "genbank" / "tp53_human.gb"
tp53_record = SeqIO.read(tp53_gb_path, "genbank")
print(f"SeqIO.read single record: {tp53_record.id} ({len(tp53_record.seq)} bp)")

# 2. SeqIO.parse streaming multi-FASTA file
multi_fasta_path = DATA_DIR / "sequences" / "multi_sample.fasta"
count = 0
total_len = 0
for record in SeqIO.parse(multi_fasta_path, "fasta"):
    count += 1
    total_len += len(record.seq)

print(f"SeqIO.parse streamed    : {count} records, {total_len:,} total bp")


#### ✏️ Try it 4.4: Find Longest Record with `SeqIO.parse`
Stream `multi_sample.fasta` using `SeqIO.parse` and identify the record with the maximum sequence length.


In [ ]:
# Try it 4.4: Longest sequence in multi_sample.fasta
longest_rec = max(SeqIO.parse(DATA_DIR / "sequences" / "multi_sample.fasta", "fasta"), key=lambda r: len(r.seq))

print(f"Longest record: {longest_rec.id} with {len(longest_rec.seq)} bp")
assert longest_rec.id == "sample_08", f"Expected sample_08, got {longest_rec.id}"
assert len(longest_rec.seq) == 449, f"Expected 449 bp, got {len(longest_rec.seq)}"
print("Try it 4.4 passed ✔")


#### ⚠️ Common Mistakes with `SeqIO`
- **Calling `SeqIO.read()` on multi-FASTA**: Raises `ValueError: More than one record found in handle`.
- **Iterating a generator twice**: `parser = SeqIO.parse(...)` is a single-use generator. Once consumed, a second loop over `parser` yields zero records. You must call `SeqIO.parse()` again or store records in a list.


### 4.5 Concept 5: Writing Sequences with `SeqIO.write()`

To export sequences to disk, use `SeqIO.write(records, handle_or_path, format)`:
- `records`: An iterable (list, tuple, generator) of `SeqRecord` objects.
- `handle_or_path`: A file path string, `pathlib.Path`, or open file handle.
- `format`: Target format (e.g. `"fasta"`, `"genbank"`, `"tab"`, `"clustal"`).
- Returns: `int` representing the total number of records successfully written.


In [ ]:
# Worked Example 4.5: Filtering and exporting SeqRecords
out_filtered_fasta = OUT_DIR / "filtered_biopython.fasta"

# Filter records with length >= 250 bp
long_records = [
    rec for rec in SeqIO.parse(DATA_DIR / "sequences" / "multi_sample.fasta", "fasta")
    if len(rec.seq) >= 250
]

written = SeqIO.write(long_records, out_filtered_fasta, "fasta")
print(f"Successfully wrote {written} records to {out_filtered_fasta.name}")


#### ✏️ Try it 4.5: Write Custom Records
Create a list containing two `SeqRecord` objects and write them to `OUT_DIR / "custom_test.fasta"` using `SeqIO.write()`.


In [ ]:
# Try it 4.5: Write SeqRecords
custom_recs = [
    SeqRecord(Seq("ATGCATGC"), id="seq_1", description="first"),
    SeqRecord(Seq("GGCCGGCC"), id="seq_2", description="second")
]
out_p = OUT_DIR / "custom_test.fasta"
n_written = SeqIO.write(custom_recs, out_p, "fasta")

assert n_written == 2, f"Expected 2 written, got {n_written}"
assert out_p.exists()
print("Try it 4.5 passed ✔")


#### ⚠️ Common Mistakes with `SeqIO.write()`
- **Passing a single `SeqRecord` instead of an iterable**: Writing `SeqIO.write(single_record, ...)` raises `TypeError` because `SeqRecord` is not iterable. Pass `[single_record]`.
- **Writing to GenBank format without required annotations**: GenBank format requires `molecule_type` in `record.annotations`. Writing an incomplete record to `"genbank"` raises `ValueError`.


### 4.6 Concept 6: GenBank Format Deep-Dive

The **GenBank** flatfile format is the richest biological record format, maintaining detailed annotations, publications, taxonomy, and gene coordinates alongside raw sequence.

#### Accessing GenBank Metadata
When parsed with `SeqIO.read()` or `SeqIO.parse(..., "genbank")`, the GenBank header information is parsed into the `.annotations` dictionary:
- `record.annotations["organism"]`: Scientific binomial organism name (e.g. `Homo sapiens`).
- `record.annotations["taxonomy"]`: Full taxonomic classification list (e.g. `['Eukaryota', 'Metazoa', 'Chordata', ...]`).
- `record.annotations["date"]`: Date of last NCBI modification.
- `record.annotations["source"]`: Biological source description.


In [ ]:
# Worked Example 4.6: Exploring GenBank annotations
tp53_rec = SeqIO.read(DATA_DIR / "genbank" / "tp53_human.gb", "genbank")

print(f"Record Accession : {tp53_rec.id}")
print(f"Definition       : {tp53_rec.description}")
print(f"Organism         : {tp53_rec.annotations.get('organism')}")
print(f"Taxonomy         : {' > '.join(tp53_rec.annotations.get('taxonomy', [])[:5])}...")
print(f"Molecule type    : {tp53_rec.annotations.get('molecule_type')}")
print(f"Total features   : {len(tp53_rec.features)}")


#### ✏️ Try it 4.6: Extract Taxonomy from Bacterial GenBank
Read `DATA_DIR / "genbank" / "ecoli_lac_operon.gb"` using `SeqIO.read` and verify that the organism is `"Escherichia coli"`.


In [ ]:
# Try it 4.6: Bacterial GenBank metadata
lac_rec = SeqIO.read(DATA_DIR / "genbank" / "ecoli_lac_operon.gb", "genbank")
organism = lac_rec.annotations.get("organism")

print(f"Organism: {organism}")
assert organism == "Escherichia coli", f"Expected Escherichia coli, got {organism}"
assert "Bacteria" in lac_rec.annotations.get("taxonomy", [])
print("Try it 4.6 passed ✔")


#### ⚠️ Common Mistakes with GenBank Files
- **Assuming keys always exist**: Not all GenBank files include every annotation field. Always use `.get("key", default)` to avoid `KeyError`.
- **Parsing `.gp` protein GenBank files as nucleotide**: Files ending in `.gp` (GenPept) contain amino acid sequences, not nucleotides.


### 4.7 Concept 7: Feature Extraction & Coding Sequences (CDS)

The `.features` attribute of a `SeqRecord` contains a list of `Bio.SeqFeature.SeqFeature` objects representing biological landmarks on the chromosome:
- `feature.type`: The feature category (`"CDS"`, `"gene"`, `"mRNA"`, `"exon"`, `"regulatory"`, `"source"`).
- `feature.location`: Exact chromosomal coordinates (`start`, `end`, `strand`).
  - `start` and `end` use **0-based, half-open** Python coordinates `[start:end)`.
  - `strand`: `+1` for forward strand, `-1` for reverse strand.
- `feature.qualifiers`: A dictionary of lists containing annotation tags:
  - `feature.qualifiers.get("gene", ["unknown"])[0]`
  - `feature.qualifiers.get("product", ["unknown"])[0]`
  - `feature.qualifiers.get("translation", [""])[0]`
- **`feature.extract(record.seq)`**: The ultimate feature extraction method! Automatically handles:
  - Coordinate slicing
  - Multi-exon compound locations (`join(100..200, 300..400)`)
  - Reverse complementation if `strand == -1`.


In [ ]:
# Worked Example 4.7: Extracting all CDS features from the E. coli lac operon
lac_rec = SeqIO.read(DATA_DIR / "genbank" / "ecoli_lac_operon.gb", "genbank")

print(f"Examining features in {lac_rec.id}:")
cds_features = [f for f in lac_rec.features if f.type == "CDS"]

for f in cds_features:
    gene = f.qualifiers.get("gene", ["unknown"])[0]
    product = f.qualifiers.get("product", ["unknown"])[0]
    start = int(f.location.start)
    end = int(f.location.end)
    strand = "+" if f.location.strand == 1 else "-"
    extracted_dna = f.extract(lac_rec.seq)
    
    print(f"Gene: {gene:6s} | {start:5d}..{end:5d} ({strand}) | Product: {product[:35]:35s} | CDS length: {len(extracted_dna)} bp")


#### ✏️ Try it 4.7: Extract lacZ Coding Sequence
Extract the CDS feature for gene `lacZ` from `ecoli_lac_operon.gb`, verify its nucleotide length, and confirm it translates cleanly.


In [ ]:
# Try it 4.7: Extract lacZ CDS
lac_rec = SeqIO.read(DATA_DIR / "genbank" / "ecoli_lac_operon.gb", "genbank")
lacz_feature = next(f for f in lac_rec.features if f.type == "CDS" and f.qualifiers.get("gene") == ["lacZ"])

lacz_dna = lacz_feature.extract(lac_rec.seq)
print(f"lacZ CDS length: {len(lacz_dna)} bp")
assert len(lacz_dna) == 3075, f"Expected 3075 bp, got {len(lacz_dna)}"

# Translate with bacterial table 11
lacz_prot = lacz_dna.translate(table=11, to_stop=True)
assert len(lacz_prot) == 1024, f"Expected 1024 amino acids, got {len(lacz_prot)}"
print("Try it 4.7 passed ✔")


#### ⚠️ Common Mistakes with Features
- **Treating qualifier values as strings**: `feature.qualifiers["gene"]` is a Python **`list`** of strings (e.g. `["lacZ"]`). Writing `f.qualifiers["gene"].upper()` raises `AttributeError: 'list' object has no attribute 'upper'`. Always index `f.qualifiers["gene"][0]`.
- **Manually slicing reverse strand features**: If a CDS is on the reverse strand (`strand == -1`), slicing `record.seq[start:end]` gives the wrong strand! Using `feature.extract(record.seq)` automatically reverse-complements correctly.


### 4.8 Concept 8: Sequence Utilities (`Bio.SeqUtils` & `ProtParam`)

Biopython provides built-in utilities for rapid biochemical and thermodynamic profiling:

#### Nucleotide Utilities (`Bio.SeqUtils`)
- `SeqUtils.gc_fraction(seq)`: Computes GC fraction as a float between `0.0` and `1.0` (modern replacement for `GC()`).
- `SeqUtils.molecular_weight(seq, seq_type="DNA")`: Computes molecular weight in Daltons ($g/mol$).
- `SeqUtils.MeltingTemp.Tm_NN(seq)`: Thermodynamic nearest-neighbor melting temperature ($T_m$) calculations.

#### Protein Physicochemical Analysis (`Bio.SeqUtils.ProtParam`)
The `ProteinAnalysis` class provides comprehensive biochemical profiling of polypeptide chains:
- `.molecular_weight()`: Accurate molecular mass of the polypeptide.
- `.isoelectric_point()`: Theoretical isoelectric point ($pI$, pH at which net charge is zero).
- `.instability_index()`: Guruprasad index. Values $< 40$ indicate a stable protein; $> 40$ indicates instability in vitro.
- `.gravy()`: Grand Average of Hydropathicity (positive = hydrophobic, negative = hydrophilic).
- `.aromaticity()`: Frequency of Phe, Tyr, and Trp residues.


In [ ]:
# Worked Example 4.8: Analyzing human insulin protein properties
insulin_gp_path = DATA_DIR / "genbank" / "insulin_protein.gp"
insulin_rec = SeqIO.read(insulin_gp_path, "genbank")
insulin_aa_str = str(insulin_rec.seq)

print(f"Protein Accession : {insulin_rec.id}")
print(f"Description       : {insulin_rec.description}")
print(f"Sequence length   : {len(insulin_aa_str)} amino acids")

# Physicochemical analysis
analyzed_protein = ProteinAnalysis(insulin_aa_str)
mw = analyzed_protein.molecular_weight()
pi = analyzed_protein.isoelectric_point()
instability = analyzed_protein.instability_index()
gravy = analyzed_protein.gravy()

print(f"Molecular Weight  : {mw:,.2f} Da")
print(f"Isoelectric Point : {pi:.2f}")
print(f"Instability Index : {instability:.2f} ({'Stable' if instability < 40 else 'Unstable'})")
print(f"GRAVY Score       : {gravy:.3f} ({'Hydrophobic' if gravy > 0 else 'Hydrophilic'})")


#### ✏️ Try it 4.8: Nucleotide GC Fraction
Calculate the GC fraction of TP53 mRNA (`tp53_rec.seq`) using `Bio.SeqUtils.gc_fraction`.


In [ ]:
# Try it 4.8: GC fraction
tp53_gc = SeqUtils.gc_fraction(tp53_rec.seq)
print(f"TP53 mRNA GC fraction: {tp53_gc:.4f} ({tp53_gc * 100:.2f}%)")

assert 0.50 <= tp53_gc <= 0.60, f"Expected GC fraction between 0.50 and 0.60, got {tp53_gc}"
print("Try it 4.8 passed ✔")


#### ⚠️ Common Mistakes with SeqUtils
- **Passing stop codons (`*`) to `ProteinAnalysis`**: If your translated string ends with `*`, `ProteinAnalysis` will crash with `KeyError: '*'`. Strip stop codons (`seq_str.rstrip('*')`) before analysis.
- **Using deprecated `SeqUtils.GC`**: `Bio.SeqUtils.GC` returned a percentage (`0-100`) and is deprecated. `Bio.SeqUtils.gc_fraction` returns a fraction (`0.0-1.0`).


### 4.9 Concept 9: File Format Conversions with `SeqIO.convert()`

Bioinformatics pipelines frequently require converting files between formats (e.g. converting GenBank records to FASTA for BLAST, or converting FASTQ to FASTA).

Instead of parsing records into memory and writing them out manually, Biopython provides `SeqIO.convert()`:
```python
converted_count = SeqIO.convert(in_file, in_format, out_file, out_format)
```
- Operates via streaming with minimal memory overhead.
- Returns the total number of records converted.


In [ ]:
# Worked Example 4.9: Converting GenBank to FASTA
gb_in = DATA_DIR / "genbank" / "tp53_human.gb"
fasta_out = OUT_DIR / "tp53_converted.fasta"

num_converted = SeqIO.convert(gb_in, "genbank", fasta_out, "fasta")
print(f"Converted {num_converted} record(s) from GenBank to FASTA.")

with open(fasta_out, "r", encoding="utf-8") as f:
    print("First 3 lines of converted file:")
    for _ in range(3):
        print(" ", repr(f.readline().strip()))


#### ✏️ Try it 4.9: Convert FASTQ to FASTA
Convert `reads_small.fastq` to FASTA format in `OUT_DIR / "reads_converted.fasta"` using `SeqIO.convert()`.


In [ ]:
# Try it 4.9: FASTQ to FASTA conversion
fastq_src = DATA_DIR / "sequences" / "reads_small.fastq"
fasta_dest = OUT_DIR / "reads_converted.fasta"

n_converted = SeqIO.convert(fastq_src, "fastq", fasta_dest, "fasta")
print(f"Converted {n_converted} reads.")
assert n_converted == 20, f"Expected 20 reads converted, got {n_converted}"
assert fasta_dest.exists()
print("Try it 4.9 passed ✔")


---
## 5. Practice Exercises

Apply your Biopython skills to solve these bioinformatics exercises:
- **Level 1 (★)**: Metadata extraction and DNA translation.
- **Level 2 (★★)**: CDS extraction, GC filtering, and protein physicochemical profiling.
- **Level 3 (★★★)**: Comprehensive Bacterial Operon CDS Exporter pipeline.

---


### Ex 2C.1 (★) — Extract GenBank Record Metadata Summary

**Problem Statement:**  
Write a function `get_genbank_summary(gb_path: Path) -> dict[str, str | int | list[str]]` that:
1. Reads a single-record GenBank file.
2. Extracts:
   - `"id"`: Record accession ID.
   - `"organism"`: Binomial organism name from annotations.
   - `"taxonomy"`: List of taxonomic categories from annotations.
   - `"seq_length"`: Length of nucleotide sequence in base pairs.
   - `"feature_count"`: Total number of features in the record.
3. Returns a dictionary containing these 5 keys.

- **Input:** `gb_path: Path`.
- **Output:** `dict` with keys `id`, `organism`, `taxonomy`, `seq_length`, `feature_count`.


In [ ]:
def get_genbank_summary(gb_path: Path) -> dict[str, str | int | list[str]]:
    rec = SeqIO.read(gb_path, "genbank")
    return {
        "id": rec.id,
        "organism": rec.annotations.get("organism", "Unknown"),
        "taxonomy": rec.annotations.get("taxonomy", []),
        "seq_length": len(rec.seq),
        "feature_count": len(rec.features)
    }


**Explanation:**
- `SeqIO.read()` loads the single GenBank record into a `SeqRecord` object.
- Annotations dictionary stores metadata such as organism and taxonomy.
- `len(rec.features)` returns the count of annotated features on the record.


In [ ]:
# ✅ Test Ex 2C.1
gb_summary = get_genbank_summary(DATA_DIR / "genbank" / "tp53_human.gb")

assert gb_summary["id"] == "NM_000546.6", f"Expected NM_000546.6, got {gb_summary['id']}"
assert gb_summary["organism"] == "Homo sapiens"
assert "Primates" in gb_summary["taxonomy"]
assert gb_summary["seq_length"] == 2512
assert gb_summary["feature_count"] > 10
print("Summary extracted:", gb_summary)
print("Ex 2C.1 passed ✔")


### Ex 2C.2 (★) — Batch Translate DNA FASTA to Protein FASTA

**Problem Statement:**  
Write a function `translate_dna_fasta_biopython(fasta_path: Path, out_path: Path, table: int = 1) -> int` that:
1. Streams DNA sequences from `fasta_path` using `SeqIO.parse()`.
2. Translates each DNA sequence to protein using the specified genetic code `table` (stopping at the first stop codon via `to_stop=True`).
3. Retains the original sequence `id` and appends `" [translated protein]"` to `description`.
4. Writes the translated records to `out_path` in FASTA format using `SeqIO.write()`.
5. Returns the total number of records translated and written.

- **Input:** `fasta_path: Path`, `out_path: Path`, `table: int`.
- **Output:** `int` total records written.


In [ ]:
def translate_dna_fasta_biopython(fasta_path: Path, out_path: Path, table: int = 1) -> int:
    protein_records = []
    for rec in SeqIO.parse(fasta_path, "fasta"):
        prot_seq = rec.seq.translate(table=table, to_stop=True)
        prot_rec = SeqRecord(
            seq=prot_seq,
            id=rec.id,
            description=f"{rec.description} [translated protein]"
        )
        protein_records.append(prot_rec)
        
    return SeqIO.write(protein_records, out_path, "fasta")


**Explanation:**
- `rec.seq.translate(table=table, to_stop=True)` ensures translation stops at the first stop codon.
- Constructing new `SeqRecord` objects preserves record IDs while updating descriptions.
- `SeqIO.write()` automatically formats the output into standard FASTA.


In [ ]:
# ✅ Test Ex 2C.2
prot_out = OUT_DIR / "translated_samples.fasta"
n_trans = translate_dna_fasta_biopython(DATA_DIR / "sequences" / "multi_sample.fasta", prot_out, table=1)

assert n_trans == 15, f"Expected 15 records translated, got {n_trans}"
assert prot_out.exists()
# Verify first translated record
first_prot = next(SeqIO.parse(prot_out, "fasta"))
assert first_prot.id == "sample_01"
assert len(first_prot.seq) > 0
print("Ex 2C.2 passed ✔")


### Ex 2C.3 (★★) — Extract All CDS Features to SeqRecords

**Problem Statement:**  
Write a function `extract_cds_features_to_records(gb_path: Path) -> list[SeqRecord]` that:
1. Reads a GenBank file.
2. Iterates through all features where `feature.type == "CDS"`.
3. Extracts the nucleotide sequence of the CDS using `feature.extract(record.seq)`.
4. Creates a new `SeqRecord` for each CDS where:
   - `id`: Gene symbol from qualifiers (`f.qualifiers.get("gene", [f.id])[0]`). If missing, use `"CDS_{start}_{end}"`.
   - `description`: Product name from qualifiers (`f.qualifiers.get("product", ["hypothetical protein"])[0]`).
   - `annotations["strand"]`: `"+" ` if strand is 1 else `"-"`.
5. Returns a list of these `SeqRecord` objects.

- **Input:** `gb_path: Path`.
- **Output:** `list[SeqRecord]`.


In [ ]:
def extract_cds_features_to_records(gb_path: Path) -> list[SeqRecord]:
    rec = SeqIO.read(gb_path, "genbank")
    extracted_records = []

    for f in rec.features:
        if f.type != "CDS":
            continue
        
        cds_dna = f.extract(rec.seq)
        start_coord = int(f.location.start)
        end_coord = int(f.location.end)
        strand_sym = "+" if f.location.strand == 1 else "-"
        
        gene_name = f.qualifiers.get("gene", [f"CDS_{start_coord}_{end_coord}"])[0]
        product_name = f.qualifiers.get("product", ["hypothetical protein"])[0]
        
        new_rec = SeqRecord(
            seq=cds_dna,
            id=gene_name,
            description=product_name,
            annotations={"strand": strand_sym, "start": start_coord, "end": end_coord}
        )
        extracted_records.append(new_rec)

    return extracted_records


**Explanation:**
- `feature.extract(rec.seq)` accurately extracts spliced exons and handles reverse complementing if on the negative strand.
- Guarding qualifiers with `.get("gene", [...])` guarantees code safety even on unannotated genomes.
- Returning a list of `SeqRecord` objects enables direct export via `SeqIO.write()`.


In [ ]:
# ✅ Test Ex 2C.3
cds_list = extract_cds_features_to_records(DATA_DIR / "genbank" / "ecoli_lac_operon.gb")

assert len(cds_list) == 4, f"Expected 4 CDS features (lacI, lacZ, lacY, lacA), got {len(cds_list)}"
gene_ids = [r.id for r in cds_list]
assert "lacZ" in gene_ids
assert "lacI" in gene_ids
print("Extracted CDS genes:", gene_ids)
print("Ex 2C.3 passed ✔")


### Ex 2C.4 (★★) — Filter FASTA by GC Fraction using `SeqUtils`

**Problem Statement:**  
Write a function `filter_fasta_by_gc_sequtils(fasta_path: Path, min_gc: float, max_gc: float) -> list[str]` that:
1. Streams records from `fasta_path` using `SeqIO.parse()`.
2. Computes the GC fraction of each sequence using modern `Bio.SeqUtils.gc_fraction()`.
3. Selects records whose GC fraction satisfies $\text{min\_gc} \le \text{gc} \le \text{max\_gc}$.
4. Returns a list of matching sequence IDs.

- **Input:** `fasta_path: Path`, `min_gc: float`, `max_gc: float` (e.g. `0.40` to `0.55`).
- **Output:** `list[str]`.


In [ ]:
def filter_fasta_by_gc_sequtils(fasta_path: Path, min_gc: float, max_gc: float) -> list[str]:
    matching_ids = []
    for rec in SeqIO.parse(fasta_path, "fasta"):
        gc = SeqUtils.gc_fraction(rec.seq)
        if min_gc <= gc <= max_gc:
            matching_ids.append(rec.id)
    return matching_ids


**Explanation:**
- `Bio.SeqUtils.gc_fraction` returns a floating-point value between `0.0` and `1.0`.
- Streaming with `SeqIO.parse()` guarantees scalability regardless of file size.


In [ ]:
# ✅ Test Ex 2C.4
matched = filter_fasta_by_gc_sequtils(DATA_DIR / "sequences" / "multi_sample.fasta", 0.20, 0.30)
assert len(matched) > 0, "Should find records with GC between 0.20 and 0.30"
assert isinstance(matched, list)
assert all(isinstance(i, str) for i in matched)
print(f"Matched {len(matched)} records within GC range [0.20, 0.30]: {matched}")
print("Ex 2C.4 passed ✔")


### Ex 2C.5 (★★) — Analyze Protein Physicochemical Properties

**Problem Statement:**  
Write a function `analyze_protein_physicochemical_properties(aa_seq: str) -> dict[str, float]` that:
1. Takes a protein sequence string `aa_seq`.
2. Removes any terminal stop codons (`*`) or trailing whitespace.
3. Instantiates `Bio.SeqUtils.ProtParam.ProteinAnalysis`.
4. Computes:
   - `"molecular_weight"`: molecular weight rounded to 2 decimals.
   - `"isoelectric_point"`: theoretical pI rounded to 2 decimals.
   - `"instability_index"`: instability index rounded to 2 decimals.
   - `"gravy"`: grand average of hydropathicity rounded to 3 decimals.
   - `"aromaticity"`: aromatic residue fraction rounded to 3 decimals.
5. Returns a dictionary containing these 5 metrics.

- **Input:** `aa_seq: str`.
- **Output:** `dict[str, float]`.


In [ ]:
def analyze_protein_physicochemical_properties(aa_seq: str) -> dict[str, float]:
    clean_seq = aa_seq.strip().rstrip("*").upper()
    pa = ProteinAnalysis(clean_seq)
    
    return {
        "molecular_weight": round(pa.molecular_weight(), 2),
        "isoelectric_point": round(pa.isoelectric_point(), 2),
        "instability_index": round(pa.instability_index(), 2),
        "gravy": round(pa.gravy(), 3),
        "aromaticity": round(pa.aromaticity(), 3)
    }


**Explanation:**
- Stripping terminal stop codons (`.rstrip("*")`) prevents `KeyError` inside `ProteinAnalysis`.
- `pa.molecular_weight()`, `pa.isoelectric_point()`, and `pa.instability_index()` provide standard biochemical characterizations.


In [ ]:
# ✅ Test Ex 2C.5
ins_rec = SeqIO.read(DATA_DIR / "genbank" / "insulin_protein.gp", "genbank")
props = analyze_protein_physicochemical_properties(str(ins_rec.seq))

assert 11000.0 < props["molecular_weight"] < 13000.0, f"Unexpected MW: {props['molecular_weight']}"
assert 5.0 < props["isoelectric_point"] < 7.0, f"Unexpected pI: {props['isoelectric_point']}"
assert isinstance(props["instability_index"], float)
print("Protein properties calculated:", props)
print("Ex 2C.5 passed ✔")


### Ex 2C.6 (★★★ Challenge) — Bacterial Operon CDS Exporter

**Problem Statement:**  
In bacterial genomics, annotating an operon requires extracting every protein-coding sequence (CDS), translating it into protein, and exporting both sequence data and a summary metadata table for publication.

Write a function `bacterial_operon_cds_exporter(gb_path: Path, out_fasta: Path, out_tsv: Path) -> int` that:
1. Reads a bacterial GenBank file (e.g. `ecoli_lac_operon.gb`).
2. Iterates over all features with `feature.type == "CDS"`.
3. For each CDS feature:
   - Extracts the nucleotide sequence using `feature.extract(record.seq)`.
   - Translates the CDS to protein using bacterial genetic code `table=11` (with `to_stop=True`).
   - Retrieves:
     - `gene`: `f.qualifiers.get("gene", ["unknown"])[0]`
     - `product`: `f.qualifiers.get("product", ["unknown"])[0]`
     - `start`: 0-based coordinate `int(f.location.start)`
     - `end`: 0-based coordinate `int(f.location.end)`
     - `strand`: `"+"` if strand is 1 else `"-"`
     - `length_nt`: length of extracted DNA sequence
     - `length_aa`: length of translated protein sequence
     - `gc_pct`: GC percentage of extracted DNA sequence rounded to 2 decimals using `SeqUtils.gc_fraction`.
4. Writes all translated protein sequences to `out_fasta` with `>gene product` headers.
5. Writes a tab-delimited table to `out_tsv` containing headers:
   `gene\tproduct\tstart\tend\tstrand\tlength_nt\tlength_aa\tgc_pct`.
6. Returns the total count of CDS features exported.

- **Input:** `gb_path: Path`, `out_fasta: Path`, `out_tsv: Path`.
- **Output:** `int` total CDS count exported.


In [ ]:
def bacterial_operon_cds_exporter(gb_path: Path, out_fasta: Path, out_tsv: Path) -> int:
    rec = SeqIO.read(gb_path, "genbank")
    cds_features = [f for f in rec.features if f.type == "CDS"]
    
    protein_records = []
    tsv_rows = []
    
    for f in cds_features:
        gene = f.qualifiers.get("gene", ["unknown"])[0]
        product = f.qualifiers.get("product", ["unknown"])[0]
        start = int(f.location.start)
        end = int(f.location.end)
        strand = "+" if f.location.strand == 1 else "-"
        
        cds_dna = f.extract(rec.seq)
        prot_seq = cds_dna.translate(table=11, to_stop=True)
        gc_val = round(SeqUtils.gc_fraction(cds_dna) * 100.0, 2)
        
        # Build SeqRecord for protein FASTA
        prot_rec = SeqRecord(
            seq=prot_seq,
            id=gene,
            description=product
        )
        protein_records.append(prot_rec)
        
        # Build TSV row
        tsv_rows.append(f"{gene}\t{product}\t{start}\t{end}\t{strand}\t{len(cds_dna)}\t{len(prot_seq)}\t{gc_val}\n")
        
    # Write protein FASTA
    SeqIO.write(protein_records, out_fasta, "fasta")
    
    # Write TSV table
    with open(out_tsv, "w", encoding="utf-8") as f_tsv:
        f_tsv.write("gene\tproduct\tstart\tend\tstrand\tlength_nt\tlength_aa\tgc_pct\n")
        for row in tsv_rows:
            f_tsv.write(row)
            
    return len(cds_features)


**Explanation:**
- `feature.extract(rec.seq)` correctly handles negative strand coordinates.
- Bacterial genetic code `table=11` translates alternate start codons correctly.
- `SeqIO.write` generates valid FASTA lines, while the TSV output formats the annotations for downstream spreadsheet or pandas analysis.


In [ ]:
# ✅ Test Ex 2C.6
out_operon_fasta = OUT_DIR / "lac_operon_proteins.fasta"
out_operon_tsv = OUT_DIR / "lac_operon_annotations.tsv"

count_exp = bacterial_operon_cds_exporter(
    DATA_DIR / "genbank" / "ecoli_lac_operon.gb",
    out_operon_fasta,
    out_operon_tsv
)

assert count_exp == 4, f"Expected 4 CDS exported, got {count_exp}"
assert out_operon_fasta.exists()
assert out_operon_tsv.exists()

# Verify exported proteins
exported_prots = list(SeqIO.parse(out_operon_fasta, "fasta"))
assert len(exported_prots) == 4
gene_names = [p.id for p in exported_prots]
assert set(gene_names) == {"lacI", "lacZ", "lacY", "lacA"}

# Verify TSV structure
with open(out_operon_tsv, "r", encoding="utf-8") as f:
    tsv_lines = f.readlines()
assert len(tsv_lines) == 5  # header + 4 rows
assert tsv_lines[0].startswith("gene\tproduct")

print("Ex 2C.6 passed ✔")


---
## 6. Summary & Cheat Sheet

| Task | Syntax | Module | Purpose |
|---|---|---|---|
| DNA $\rightarrow$ Reverse Complement | `seq.reverse_complement()` | `Bio.Seq` | 5′ $\rightarrow$ 3′ reverse complement |
| DNA $\rightarrow$ mRNA | `seq.transcribe()` | `Bio.Seq` | T $\rightarrow$ U transcription |
| DNA $\rightarrow$ Protein | `seq.translate(table=1, to_stop=True)` | `Bio.Seq` | Translation with genetic code |
| Read Single Record | `rec = SeqIO.read(path, "genbank")` | `Bio.SeqIO` | Exactly 1 record |
| Stream Multi-Record | `for rec in SeqIO.parse(path, "fasta"):` | `Bio.SeqIO` | $O(1)$ memory streaming |
| Write Records | `SeqIO.write(records_list, path, "fasta")` | `Bio.SeqIO` | Export SeqRecords |
| Format Conversion | `SeqIO.convert(in_f, "gb", out_f, "fasta")` | `Bio.SeqIO` | Streaming inter-conversion |
| Extract Feature DNA | `feature.extract(record.seq)` | `Bio.SeqFeature` | Handles splicing & strand automatically |
| Compute GC Fraction | `SeqUtils.gc_fraction(seq)` | `Bio.SeqUtils` | Returns float `0.0` to `1.0` |
| Protein Analysis | `pa = ProteinAnalysis(aa_str)` | `Bio.SeqUtils.ProtParam` | Computes MW, pI, instability, GRAVY |

---


## 7. Further Reading & Resources

- [Biopython Official Documentation & Tutorial](https://biopython.org/DIST/docs/tutorial/Tutorial.html)
- [Biopython `SeqIO` Format Wiki](https://biopython.org/wiki/SeqIO)
- [NCBI Genetic Code Tables (Bacterial Table 11 vs Standard Table 1)](https://www.ncbi.nlm.nih.gov/Taxonomy/Utils/wprintgc.cgi)
- [Guruprasad et al. (1990): Correlation between protein stability and amino acid composition (Instability Index)](https://pubmed.ncbi.nlm.nih.gov/2075190/)
